In [1]:
import warnings, time
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

# LogisticRegression
from sklearn.linear_model import LogisticRegression

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity


# add path to project root
import sys
PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))
if PROJECT_ROOT.name == "notebooks":
    sys.path.append(str(PROJECT_ROOT.parent))
    
    
from src.preprocess.gaming_text_preprocessor import GamingTextPreprocessor
from src.preprocess.gaming_text_labeler import GamingTextLabeler


# instantiate preprocessor and labeler
gaming_preprocessor = GamingTextPreprocessor()
gaming_labeler = GamingTextLabeler()

In [ ]:
# config
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'processed_data').exists() and PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label'
}

np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

print('CONFIG loaded. Text column:', tc)


CONFIG loaded. Text column: message


# Fit the Gaming Text Preprocessor 

In [3]:
# load WOT train data
d = CONFIG['data_dir']

wot_train = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train["data_source"] = "wot"

In [4]:
# load DOTA training data
dota_train = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train["data_source"] = "dota"

In [5]:
# combine datasets
train = pd.concat([wot_train, dota_train], ignore_index=True)

In [6]:
# train 
gaming_preprocessor.fit(train, text_column=tc)

# Label Cleaning

In [9]:
# convert dota labels 

train = gaming_labeler.convert_three_class(
    train, 
    label_column=lc, 
    data_source_column='data_source',
    output_column='label_3class'
)


In [10]:
train["label_3class"].value_counts()

label_3class
0    40929
1     7492
2     5286
Name: count, dtype: int64